# GPU SAT solver benchmark for L-meta-1988 CNFs

Runtime > Change runtime type > **GPU (T4)** first.

Compares on the same CNFs (exact monotone-circuit encoding, cycle graphs C8/C10, all UNSAT):
- kissat (CPU, 1 thread), baseline
- ParaFROST CPU vs ParaFROST GPU (GPU inprocessing)
- glucose-syrup (CPU clause sharing) vs GpuShareSat glucose-gpu (GPU clause sharing), same thread count

Each run has a 600 s limit. Results land in `results_gpu.tsv`; please send that file back.

In [ ]:
!nvidia-smi
!nproc
!free -g | head -2
!nvcc --version | tail -1

In [ ]:
!pip -q install python-sat
!mkdir -p /content/w/bench

In [ ]:
%%writefile /content/w/exact_enc.py
#!/usr/bin/env python3
# Exact SAT model: is there a monotone fan-in-2 AND/OR circuit with s gates
# computing f_G (G bipartite), optionally with a free final AND of <= 2 wires?
# Correctness is checked only on minterms (edges) and maximal false points
# (maximal independent sets, as (P',L') pairs) -- sufficient for monotone circuits.
import sys, itertools, argparse
from pysat.formula import CNF, IDPool
from pysat.card import CardEnc, EncType

def plane(q):
    if q == 2:
        m = 7; D = [0, 1, 3]
    elif q == 3:
        m = 13; D = [0, 1, 3, 9]
    else:
        raise ValueError
    inc = lambda p, l: (p + l) % m in D
    return m, inc

def rows_for(m, inc):
    n = 2 * m
    pos = []
    for p in range(m):
        for l in range(m):
            if inc(p, l):
                v = [0] * n; v[p] = 1; v[m + l] = 1; pos.append(v)
    # maximal independent pairs (P',L') with L'=avoid(P'), P'=avoid(L')
    neg = set()
    for mask in range(1 << m):
        P = [p for p in range(m) if mask >> p & 1]
        L = [l for l in range(m) if all(not inc(p, l) for p in P)]
        P2 = [p for p in range(m) if all(not inc(p, l) for l in L)]
        if P2 == P:
            v = [0] * n
            for p in P: v[p] = 1
            for l in L: v[m + l] = 1
            neg.add(tuple(v))
    return pos, [list(v) for v in sorted(neg)]

def build(m, inc, s, free_and=True, distinct=True, used=True, lexsb=True, and_lo=None, and_hi=None):
    n = 2 * m
    pos, neg = rows_for(m, inc)
    rows = [(r, 1) for r in pos] + [(r, 0) for r in neg]
    R = len(rows)
    pool = IDPool(); cnf = CNF()
    W = n + s
    isand = lambda i: pool.id(('and', i))
    A = lambda i, j: pool.id(('a', i, j))  # first input of gate i is wire j
    Bv = lambda i, j: pool.id(('b', i, j))  # second input of gate i is wire j
    val = lambda w, r: pool.id(('v', w, r))  # value of wire w (>= n) on row r
    ua = lambda i, r: pool.id(('ua', i, r))
    ub = lambda i, r: pool.id(('ub', i, r))

    def wv(w, r):  # literal or constant
        if w < n:
            return rows[r][0][w] == 1
        return val(w, r)

    for i in range(s):
        g = n + i
        # exactly one choice each, a < b
        cnf.extend(CardEnc.equals([A(i, j) for j in range(g)], 1, vpool=pool, encoding=EncType.seqcounter).clauses)
        cnf.extend(CardEnc.equals([Bv(i, j) for j in range(g)], 1, vpool=pool, encoding=EncType.seqcounter).clauses)
        for j in range(g):
            for k in range(j + 1):
                # forbid b <= a
                cnf.append([-A(i, j), -Bv(i, k)])
        for r in range(R):
            for j in range(g):
                x = wv(j, r)
                for sel, u in ((A(i, j), ua(i, r)), (Bv(i, j), ub(i, r))):
                    if x is True:
                        cnf.append([-sel, u])
                    elif x is False:
                        cnf.append([-sel, -u])
                    else:
                        cnf.append([-sel, -x, u]); cnf.append([-sel, x, -u])
            v = val(g, r); a = ua(i, r); b = ub(i, r); t = isand(i)
            # AND: v <-> a & b
            cnf.append([-t, -v, a]); cnf.append([-t, -v, b]); cnf.append([-t, v, -a, -b])
            # OR: v <-> a | b
            cnf.append([t, v, -a]); cnf.append([t, v, -b]); cnf.append([t, -v, a, b])

    last = n + s - 1
    if free_and:
        # outputs: last gate and o1 (any gate, possibly the last itself)
        O = lambda w: pool.id(('o', w))
        cnf.extend(CardEnc.equals([O(w) for w in range(n, n + s)], 1, vpool=pool, encoding=EncType.seqcounter).clauses)
        for r, (row, lab) in enumerate(rows):
            y = pool.id(('y', r))
            for w in range(n, n + s):
                cnf.append([-O(w), -val(w, r), y]); cnf.append([-O(w), val(w, r), -y])
            if lab == 1:
                cnf.append([val(last, r)]); cnf.append([y])
            else:
                cnf.append([-val(last, r), -y])
    else:
        for r, (row, lab) in enumerate(rows):
            cnf.append([val(last, r)] if lab == 1 else [-val(last, r)])

    if used:
        for i in range(s - 1):
            g = n + i
            lits = [A(k, g) for k in range(i + 1, s)] + [Bv(k, g) for k in range(i + 1, s)]
            if free_and:
                lits.append(pool.id(('o', g)))
            cnf.append(lits)

    if distinct:
        # all wire vectors on the rows pairwise distinct (gates vs gates and gates vs inputs)
        for i in range(s):
            g = n + i
            for j in range(g):
                d = []
                for r in range(R):
                    x = wv(j, r)
                    if x is True:
                        d.append(-val(g, r))
                    elif x is False:
                        d.append(val(g, r))
                    else:
                        e = pool.id(('d', g, j, r))
                        # e -> v_g != v_j
                        cnf.append([-e, val(g, r), x]); cnf.append([-e, -val(g, r), -x])
                        d.append(e)
                cnf.append(d)
        # no constant gates
        for i in range(s):
            g = n + i
            cnf.append([val(g, r) for r in range(R)])
            cnf.append([-val(g, r) for r in range(R)])

    if lexsb:
        # consecutive independent gates: (a_i, b_i) <=lex (a_{i+1}, b_{i+1})
        for i in range(s - 1):
            g = n + i
            dep = [A(i + 1, g), Bv(i + 1, g)]
            for j in range(g):
                for k in range(j):
                    # a_i = j, a_{i+1} = k < j  forbidden unless dependent
                    cnf.append(dep + [-A(i, j), -A(i + 1, k)])
            for j in range(g):
                for b1 in range(g):
                    for b2 in range(b1):
                        cnf.append(dep + [-A(i, j), -A(i + 1, j), -Bv(i, b1), -Bv(i + 1, b2)])
    if used:
        # every input variable is read by some gate (f depends on all of them)
        for j in range(n):
            cnf.append([A(i, j) for i in range(s)] + [Bv(i, j) for i in range(s)])
    ands = [isand(i) for i in range(s)]
    if and_lo is not None:
        cnf.extend(CardEnc.atleast(ands, and_lo, vpool=pool, encoding=EncType.seqcounter).clauses)
    if and_hi is not None:
        cnf.extend(CardEnc.atmost(ands, and_hi, vpool=pool, encoding=EncType.seqcounter).clauses)
    return cnf, pool, rows

def decode(model, pool, m, s, free_and):
    n = 2 * m
    ms = set(l for l in model if l > 0)
    out = []
    for i in range(s):
        a = [j for j in range(n + i) if pool.id(('a', i, j)) in ms][0]
        b = [j for j in range(n + i) if pool.id(('b', i, j)) in ms][0]
        t = 'AND' if pool.id(('and', i)) in ms else 'OR'
        out.append((t, a, b))
    o = None
    if free_and:
        o = [w for w in range(n, n + s) if pool.id(('o', w)) in ms][0]
    return out, o

if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('q', type=int)
    ap.add_argument('s', type=int)
    ap.add_argument('--plain', action='store_true')
    ap.add_argument('--out', default=None)
    ap.add_argument('--and-lo', type=int, default=None)
    ap.add_argument('--and-hi', type=int, default=None)
    a = ap.parse_args()
    m, inc = plane(a.q)
    cnf, pool, rows = build(m, inc, a.s, free_and=not a.plain, and_lo=a.and_lo, and_hi=a.and_hi)
    print('rows', len(rows), 'vars', pool.top, 'clauses', len(cnf.clauses), file=sys.stderr)
    cnf.to_file(a.out)


In [ ]:
%%writefile /content/w/bench_gen.py
import sys
sys.path.insert(0, sys.argv[1])
from exact_enc import build
out = sys.argv[2]
cyc = lambda k: (k, lambda p, l: l == p or l == (p + 1) % k)
for name, (m, inc), s in [('c8_s8', cyc(4), 8), ('c8_s9', cyc(4), 9), ('c8_s10', cyc(4), 10), ('c10_s10', cyc(5), 10), ('c10_s11', cyc(5), 11)]:
    cnf, pool, rows = build(m, inc, s, free_and=False)
    cnf.to_file(f'{out}/{name}.cnf')
    print(name, 'vars', pool.top, 'clauses', len(cnf.clauses))


In [ ]:
%%writefile /content/w/bench.py
#!/usr/bin/env python3
# Wall-clock benchmark of SAT solvers on the exact-circuit CNFs.
# usage: bench.py <timeout_s> <out.tsv> <name=command-with-{cnf}> ... -- <cnf> ...
import subprocess, sys, time, os, re

def run(cmd, timeout):
    t = time.time()
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        out = p.stdout + p.stderr
        dt = time.time() - t
    except subprocess.TimeoutExpired:
        return 'TIMEOUT', timeout
    out = re.sub(r'\x1b\[[0-9;]*m', '', out)
    if re.search(r'^s UNSATISFIABLE', out, re.M):
        return 'UNSAT', dt
    if re.search(r'^s SATISFIABLE', out, re.M):
        return 'SAT', dt
    return 'UNKNOWN(rc=%s)' % p.returncode, dt

def main():
    timeout = int(sys.argv[1]); tsv = sys.argv[2]
    i = sys.argv.index('--')
    solvers = [a.split('=', 1) for a in sys.argv[3:i]]
    cnfs = sys.argv[i + 1:]
    new = not os.path.exists(tsv)
    with open(tsv, 'a') as f:
        if new:
            f.write('instance\tsolver\tresult\tseconds\n')
        for c in cnfs:
            for name, tmpl in solvers:
                r, dt = run(tmpl.replace('{cnf}', c), timeout)
                line = '%s\t%s\t%s\t%.1f' % (os.path.basename(c), name, r, dt)
                print(line, flush=True); f.write(line + '\n'); f.flush()

if __name__ == '__main__':
    main()


In [ ]:
%cd /content/w
!python3 -I bench_gen.py /content/w /content/w/bench
!ls -la bench

In [ ]:
%cd /content/w
!git clone -q --depth 1 https://github.com/arminbiere/kissat.git && cd kissat && ./configure > /dev/null && make -j$(nproc) > /dev/null 2>&1 && ls -la build/kissat

In [ ]:
%cd /content/w
!git clone -q --recursive https://github.com/muhos/ParaFROST.git
%cd /content/w/ParaFROST
!./install.sh -c > cpu_build.log 2>&1; tail -2 cpu_build.log
!./install.sh -g > gpu_build.log 2>&1; tail -5 gpu_build.log
!ls build/*/bin

In [ ]:
# GPU compute capability for GpuShareSat (T4 = 75)
import subprocess
cc = subprocess.run(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader'],capture_output=True,text=True).stdout.strip().split('\n')[0].replace('.','')
print('SMVER', cc)
%cd /content/w
!git clone -q --depth 1 https://github.com/nicolasprevot/GpuShareSat.git
%cd /content/w/GpuShareSat/glucose-syrup/gpu
!sed -i 's/^SMVER=.*/SMVER={cc}/' Makefile
!make glucose-gpu_release > build.log 2>&1; tail -3 build.log; ls -la glucose-gpu_release

In [ ]:
%cd /content/w
!git clone -q --depth 1 https://github.com/audemard/glucose.git glucose-cpu
%cd /content/w/glucose-cpu/parallel
!make rs > build.log 2>&1; tail -2 build.log; ls -la glucose-syrup_static

In [ ]:
%cd /content/w
import os
T=os.cpu_count()
W='/content/w'
solvers = [
 f'kissat-1T={W}/kissat/build/kissat -q {{cnf}}',
 f'parafrost-cpu={W}/ParaFROST/build/cpu/bin/parafrost {{cnf}} -quiet',
 f'parafrost-gpu={W}/ParaFROST/build/gpu/bin/parafrost {{cnf}} -quiet',
 f'glucose-syrup-{T}T={W}/glucose-cpu/parallel/glucose-syrup_static -nthreads={T} {{cnf}}',
 f'gpusharesat-{T}T={W}/GpuShareSat/glucose-syrup/gpu/glucose-gpu_release -thread-count={T} {{cnf}}',
]
args = ' '.join(repr(s) for s in solvers)
cnfs = ' '.join(f'{W}/bench/{n}.cnf' for n in ['c8_s8','c8_s9','c10_s10','c8_s10'])
!python3 -I bench.py 600 results_gpu.tsv {args} -- {cnfs}

In [ ]:
!cat /content/w/results_gpu.tsv
from google.colab import files
files.download('/content/w/results_gpu.tsv')